In [1]:
import sys
sys.dont_write_bytecode = True
sys.path.insert(0, "..")
import numpy as np
from tqdm import tqdm
import pandas as pd
import datetime as dt
from scipy.stats import pearsonr
import pickle
import torch
import torch.nn as nn
import os
import time
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import math
from math import sqrt
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
import codes.mnn_Utils as mnn
from codes.make_dataset import DatasetHist
from codes.data_utils import ips_omni_processor
import random

from codes.plot_utils import plot_losses

from codes.train_utils import cleanup

from termcolor import colored


# device = torch.device("cuda")
device = torch.device("xpu")

%load_ext autoreload
%autoreload 2

In [2]:
val_df = pd.read_csv("../data/data_generated/val/val_ips_omni_df.csv")
val_df.shape, val_df.shape[0]/16

((5406, 433), 337.875)

In [3]:
val_df = pd.read_csv("../data/data_generated/val_corrected/val_ips_omni_df.csv")
val_df.shape, val_df.shape[0]/16

((5406, 433), 337.875)

In [5]:
train_df = pd.read_csv("../data/data_generated/train_corrected/train_ips_omni_df.csv")

In [6]:
train_df.columns.tolist()[:14]

['idx',
 'X_dist_0',
 'X_hla_0',
 'X_hlo_0',
 'X_gla_0',
 'X_glo_0',
 'X_carr_0',
 'X_v_0',
 'X_er_0',
 'X_sc_indx_0',
 'X_time_0',
 'X_day_total_0',
 'X_time_trgt_0',
 'X_input_0']

In [7]:
train_df.shape, train_df.shape[0]/16

((71992, 433), 4499.5)

In [5]:
def train_tester(df_path:str):
    train_df = pd.read_csv(df_path)

In [8]:
train_ds = DatasetHist("../data/data_generated/train_corrected/train_ips_omni_df.csv")
val_ds = DatasetHist("../data/data_generated/val_corrected/val_ips_omni_df.csv")

Input features are ['dist', 'hla', 'hlo', 'gla', 'glo', 'carr', 'v', 'er', 'sc_indx', 'time', 'day_total', 'time_trgt', 'input']
Target features are ['swSpeed_Smth_0']
begins: 0 ends: 71991
Input features are ['dist', 'hla', 'hlo', 'gla', 'glo', 'carr', 'v', 'er', 'sc_indx', 'time', 'day_total', 'time_trgt', 'input']
Target features are ['swSpeed_Smth_0']
begins: 71994 ends: 77399


In [9]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=False)
val_dl = DataLoader(val_ds, batch_size=16, shuffle=False)

In [10]:
len(val_ds)/16

337.875

In [26]:
model = mnn.model_base(xCh=13, LEN=32, emDim=128, dropout=0.2, device=device)
model = model.to(device)
model.eval();

In [28]:
def train_hyperprams(
    model,
    optimizer,
    train_path:str, 
    val_path:str, 
    test_path:str,
    epochs:int,
    batch_size:int = 16,
    train_step:int=500,
    diff_alpha:int=0.0
):
    """
    Train a model using specified hyperparameters and evaluate performance on
    training, validation, and test datasets.

    This function orchestrates the full training loop, including dataset loading,
    dataloader construction, forward and backward passes, validation at each epoch,
    and final test evaluation using correlation and MSE metrics. The function
    returns training and validation losses per epoch, along with the best test-set
    statistics observed during training.

    Parameters
    ----------
    model : torch.nn.Module
        The neural network model to be trained.
    optimizer : torch.optim.Optimizer
        Optimizer instance used for gradient updates.
    train_path : str
        Path to the training dataset file or directory compatible with
        `DatasetHist`.
    val_path : str
        Path to the validation dataset file or directory compatible with
        `DatasetHist`.
    test_path : str
        Path to the test dataset file or directory compatible with `DatasetHist`.
    epochs : int
        Number of training epochs.
    batch_size : int, optional
        Batch size used during training and validation. Defaults to ``16``.
    train_step: int =500,
        No.of. training batches to be trained upon.
    diff_alpha: int =0.0,
        alpha for difference in loss wrt time for back propagation.

    Returns
    -------
    train_Loss : numpy.ndarray of shape (epochs,)
        Scaled training loss per epoch (weighted MSE used for backpropagation).
    train_Loss_y : numpy.ndarray of shape (epochs,)
        Standard MSE training loss per epoch (unscaled).
    val_Loss : numpy.ndarray of shape (epochs,)
        Scaled validation loss per epoch.
    val_Loss_y : numpy.ndarray of shape (epochs,)
        Standard MSE validation loss per epoch.
    bestCorr : list of float
        Best Pearson correlation values for each monitored channel in the test set.
    bestMse : list of float
        Best MSE values (scaled by 800) for each monitored channel in the test set.
    bestEpoch : int
        Epoch index at which the best correlation performance was observed.

    Notes
    -----
    - Training loss used for backpropagation is scaled by the target values and a
      factor of 100. The unscaled MSE is stored separately.
    - Best test-set correlation and MSE are computed on four reference channels
      defined by indices ``[9, 11, 13, 15]``.
    - The scheduler used for learning rate adjustments is expected to be defined
      externally in the calling scope.
    - Global variables such as ``device``, ``lr``, ``weight_decay``, ``dropout``,
      ``gamma``, and ``scheduler`` must be defined outside this function.

    """

    # Load Datasets
    train_ds = DatasetHist(train_path)
    val_ds = DatasetHist(val_path)
    test_ds = DatasetHist(test_path)

    # Load DataLoader
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_dl = DataLoader(test_ds, batch_size=len(test_ds), shuffle=False)

    model = model.to(device)

    # Compute the loss derivative wrt time
    def diff_loss(loss):
        loss_len = loss.shape[0]
        loss_diff = torch.empty((loss_len - 1, 1, 16), dtype=torch.float32, device= device)
        loss_diff = loss[1:, :, :] - loss[:-1, :, :]
        return loss_diff

    # Run model on data, used both on train and val
    # Define Backprop loss here
    def doStep(data):
        x = data[1].to(torch.float32)
        x = x.to(device)
        # print("x", x)
        y = data[2].to(torch.float32)
        y = y.to(device)
        y_out = model(x)
        # print("Before backward:", torch.isnan(y_out).any())

        loss = F.mse_loss(y_out, y, reduction='none')
        loss = loss * y * 100.0 
        loss_diff = diff_loss(loss)                   # derviative of loss wrt time i.e. loss difference in a batch
        loss = loss.mean()                            # Loss used for Backprop, y scaled
        y_loss = F.mse_loss(y_out, y)                 # The actual loss: mse

        return loss, y_loss, loss_diff

    # Training for an epoch
    def train_epoch(train_dl, epoch, train_step:int=500):
        train_dl_len = len(train_dl)
        model.eval()
        running_loss = 0.0
        running_loss_y = 0.0

        loop = tqdm(enumerate(train_dl), total=train_step, leave=False)
        for i, data in loop:
            # zero the parameter gradients
            optimizer.zero_grad()
            loss, y_loss, loss_diff = doStep(data)
            # print(i, "epoch:", epoch, ";", "loss", loss.item())
            del data

            # print(loss.dtype)
            loss_bkprp = loss + diff_alpha * loss_diff
            loss_bkprp.backward()
            optimizer.step()

            running_loss += loss.item()
            running_loss_y += y_loss.item()

            # update progress bar:
            loop.set_description(f"Epoch [{epoch}/{epochs}]")
            loop.set_postfix(loss=y_loss.item(), lr=lr, weight_decay= weight_decay, batch_size=batch_size, gamma=gamma)
            if i == train_step:
                break

        # Epoch averages
        running_loss = running_loss / train_step
        running_loss_y = running_loss_y / train_step
        print("")
        print(lr, weight_decay, dropout, batch_size, gamma)
        print("Epoch: ", epoch, "Error: ", np.sqrt(running_loss_y) * 800.0)
        return running_loss, running_loss_y

    # Validation for an epoch
    def validate_epoch(val_dl, epoch):
        model.eval()
        with torch.no_grad():
            running_loss = 0.0
            running_loss_y = 0.0
            val_dl_len = len(val_dl)

            loop = tqdm(enumerate(val_dl), total=val_dl_len, leave=False)
            for i, data in loop:
                loss, y_loss, _ = doStep(data)
                # print("val loss", loss)
                del data

                running_loss += loss.item()
                running_loss_y += y_loss.item()

                loop.set_description(f"Validate [{i}/{val_dl_len}]")
                loop.set_postfix(loss=loss.item())

            # Epoch average losses 
            running_loss = running_loss / val_dl_len
            running_loss_y = running_loss_y / val_dl_len
            print("Epoch: ", epoch, "Error: ", np.sqrt(running_loss_y) * 800.0)
        return running_loss, running_loss_y

    # Test evlaution for an epoch
    def test_epoch(test_dl, epoch, bestCorr, bestMse, bestEpoch):
        model.eval()
        with torch.no_grad():
            for batch in test_dl:
                testBatch = batch
                break
            x = testBatch[1].to(torch.float32)
            x = x.to(device)
            # y = testBatch[2]
            opY = model(x).detach().cpu().numpy()

        refY = testBatch[2].numpy()
        # print("refY.shape", refY.shape)

        corrVals = [-1.0, -1.0, -1.0, -1.0]
        mseVals = [300.0, 300.0, 300.0, 300.0]
        refIds = [9, 11, 13, 15]
        for i in range(4):
            thisRef = refY[:, :, refIds[i]].flatten()
            thisOp = opY[:, :, refIds[i]].flatten()
            corrVals[i] = pearsonr(thisRef, thisOp)[0]
            mseVals[i] = np.mean(np.sqrt((thisOp - thisRef)**2) * 800.0)
        if corrVals[-1] > bestCorr[-1]:
            bestCorr = corrVals
            # torch.save(
            #     model.state_dict(), resultsPath + 'models/%s_%s' %
            #     (mString, paraString))
            bestMse = mseVals
            bestEpoch = epoch
        print(
            "Evaluate:  ",
            " epoch:",
            epoch,
            ", bestCorr:",
            bestCorr,
            ", bestEpoch:",
            bestEpoch)
        return bestCorr, bestMse, bestEpoch


    # Initialize loss arrays
    train_Loss_y = np.empty(epochs, dtype=np.float32)
    train_Loss = np.empty(epochs, dtype=np.float32)
    val_Loss_y = np.empty(epochs, dtype=np.float32)
    val_Loss = np.empty(epochs, dtype=np.float32)

    # Initialize Bests
    bestCorr = [-1.0, -1.0, -1.0, -1.0]
    print('bestCorr is defined')
    bestMse = [999.0, 999.0, 999.0, 999.0]
    bestEpoch = epochs
    
    for epoch in range(1, epochs + 1):
        print(
            f"Training epoch {epoch} =================================================================================")
        tlosses = train_epoch(train_dl=train_dl, epoch=epoch, train_step=train_step)
        print(f"Validating epoch {epoch}")
        vlosses = validate_epoch(val_dl=val_dl, epoch= epoch)
        scheduler.step()

        # Scaled losses: y x 100 x mse
        train_Loss[epoch - 1] = tlosses[0]
        val_Loss[epoch - 1] = vlosses[0]

        # Usual losses: mse
        train_Loss_y[epoch - 1] = tlosses[1]
        val_Loss_y[epoch - 1] = vlosses[1]

        if epoch > 1:
            print(f"Evaluating epoch {epoch}")
            bestCorr, bestMse, bestEpoch = test_epoch(test_dl, epoch, bestCorr, bestMse, bestEpoch)

    return train_Loss, train_Loss_y, val_Loss, val_Loss_y, bestCorr, bestMse, bestEpoch


In [107]:
for lr in lrs[:1]:
    print(lr)

0.001


In [ ]:
# define model
mString = 'base_v0'
# Define learning parameters ---------------------------------------------
lrs = [1.0e-3, 1.0e-2]  # learning rate
weight_decays = [1.0e-4, 1.0e-3]  # weight decay regularization
dropouts = [0.3, 0.2, 0.4]  # dropouts
batch_sizes = [16, 32]      # batchsize
# ,0.80] #learning rate scheduler, reduces the learning rate as training gets cloes to the minima
gammas = [0.95, 0.90, 0.85]
kwargs = {'num_workers': 4, 'pin_memory': True}
torch.manual_seed(42)
resultsPath = "../model_outputs/local_runs/"
# Inititialize optimizer and scheduler

epochs = 1
startT = time.time()

for lr in lrs[:1]:
    for weight_decay in weight_decays[:1]:
        for dropout in dropouts[:1]:
            for batch_size in batch_sizes[:1]:
                for gamma in gammas[:1]:
                    # Cleanup memory
                    try:
                        cleanup(model, optimizer)
                    except NameError:
                        pass
                    # load model, optimizer and scheduler
                    model = mnn.model_base(xCh=13, LEN=32, emDim=128, dropout=0.2, device=device)
                    optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
                    scheduler = StepLR(optimizer, step_size=1, gamma=gamma)

                    # get values:
                    train_Loss, train_Loss_y, val_Loss, val_Loss_y, bestCorr, bestMse, bestEpoch = train_hyperprams(
                        model=model,
                        optimizer=optimizer,
                        train_path="../data/data_generated/train/train_ips_omni_df.csv",
                        val_path="../data/data_generated/val/val_ips_omni_df.csv",
                        test_path="../data/data_generated/test/test_ips_omni_df.csv",
                        epochs=epochs,
                        batch_size=batch_size
                    )

                    # String to store model training parameters
                    paraString = 'lr%0.2e_wd%0.2e_drp%0.2f_bS%d_E%d_sR%0.2f' % (
                        lr, weight_decay, dropout, batch_size, epochs, gamma)
                    
                    # Store losses
                    train_Loss.dump(resultsPath + 'losses/%s_train_%s' % (mString, paraString))
                    val_Loss.dump(resultsPath + 'losses/%s_val_%s' % (mString, paraString))
                    train_Loss_y.dump(resultsPath + 'losses/%s_train_y_%s' % (mString, paraString))
                    val_Loss_y.dump(resultsPath + 'losses/%s_val_y_%s' % (mString, paraString))
                    # Store correlations and statistics
                    with open(resultsPath + 'corrsM2e', 'a') as fl:
                        # fl.write('%s_%s\t%0.6f\t%0.6f\t%d\n'%(mString,paraString,bestCorr,bestMse,bestEpoch))
                        fl.write('%s_%s' % (mString, paraString))
                        for i in range(4):
                            fl.write('\t%0.6f\t%0.6f' % (bestCorr[i], bestMse[i]))
                        fl.write('\t%d\n' % bestEpoch)
                    print(lr, time.time() - startT)

In [1]:
batch_sizes = [16, 32][-1:] 

In [2]:
batch_sizes

[32]

In [3]:
aT = torch.randint(10, (5, 1, 4))
aT

tensor([[[5, 4, 7, 1]],

        [[0, 5, 4, 4]],

        [[8, 1, 9, 0]],

        [[6, 6, 9, 5]],

        [[2, 1, 4, 8]]])

In [4]:
aT[1:, :, :] - aT[:-1, :, :]

tensor([[[-5,  1, -3,  3]],

        [[ 8, -4,  5, -4]],

        [[-2,  5,  0,  5]],

        [[-4, -5, -5,  3]]])

In [62]:
aT = torch.rand(5, 1, 4)
aT

tensor([[[0.8358, 0.1270, 0.7500, 0.2246]],

        [[0.7263, 0.6709, 0.6095, 0.7814]],

        [[0.2002, 0.5896, 0.5365, 0.1521]],

        [[0.8383, 0.4694, 0.6409, 0.1937]],

        [[0.5926, 0.6910, 0.7179, 0.6819]]])

In [63]:
aT - torch.mean(aT, dim=0, keepdim=True)
aT - torch.mean(aT, dim=0, keepdim=True), torch.mean(aT - torch.mean(aT, dim=0, keepdim=True), dim=0)

(tensor([[[ 0.1972, -0.3825,  0.0991, -0.1821]],
 
         [[ 0.0876,  0.1613, -0.0414,  0.3747]],
 
         [[-0.4384,  0.0800, -0.1145, -0.2546]],
 
         [[ 0.1997, -0.0402, -0.0101, -0.2131]],
 
         [[-0.0460,  0.1814,  0.0669,  0.2751]]]),
 tensor([[0.0000e+00, 2.3842e-08, 1.1921e-08, 1.7881e-08]]))

In [64]:
torch.sum((aT - torch.mean(aT, dim=0, keepdim=True))**2, dim=0)

tensor([[0.2807, 0.2133, 0.0292, 0.3594]])

In [65]:
bT = torch.rand(5, 1, 4)
bT

tensor([[[0.4489, 0.5162, 0.9879, 0.2456]],

        [[0.5784, 0.4457, 0.9924, 0.7279]],

        [[0.8079, 0.0805, 0.6507, 0.2730]],

        [[0.0617, 0.3257, 0.9250, 0.0818]],

        [[0.7734, 0.7416, 0.9957, 0.1493]]])

In [66]:
torch.sum((aT - torch.mean(aT, dim=0, keepdim=True))**2, dim=0) * torch.sum((bT - torch.mean(bT, dim=0, keepdim=True))**2, dim=0)

tensor([[0.1024, 0.0507, 0.0026, 0.0924]])

In [67]:
cov_aT_bT = torch.sum((aT - torch.mean(aT, dim=0, keepdim=True)) * (bT - torch.mean(bT, dim=0, keepdim=True)), dim=0)
cov_aT_bT

tensor([[-0.2383,  0.0023,  0.0396,  0.1821]])

In [69]:
(1 - cov_aT_bT**2/(torch.sum((aT - torch.mean(aT, dim=0, keepdim=True))**2, dim=0) * torch.sum((bT - torch.mean(bT, dim=0, keepdim=True))**2, dim=0) + 1e-8)).mean()

tensor(0.6186)